# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, length

# Reading from Bronze Table

In [0]:
df = spark.table('workspace.bronze.erp_px_cat_g1v2')

# Data Transformations

### Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

### Normalize Maintenance flag to Boolean

In [0]:
df = df.withColumn(
    "maintenance",
    F.when(F.upper(col("maintenance")) == "YES", F.lit(True))
     .when(F.upper(col("maintenance")) == "NO", F.lit(False))
     .otherwise(None)
)

# Renaming Columns

In [0]:
renaming_map = {
    'ID': 'category_id',
    'CAT': 'category',
    'SUBCAT': 'subcategory',
    'MAINTENANCE': 'maintenance_flag'
}

# Checking DataFrame

In [0]:
df.display()

# Writing into Silver Table

In [0]:
(
    df.write
    .mode('overwrite')
    .format('delta')
    .saveAsTable('workspace.silver.erp_product_category')
)